<a href="https://colab.research.google.com/github/armandochernandez-ai/Curso-python-slava/blob/main/Desaparecidos/Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Método DIRECTO - Analizar el HTML del mapa y buscar datos
import requests
import json
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Crear carpeta
drive_folder = '/content/drive/My Drive/Desaparecidos_Jalisco'
os.makedirs(drive_folder, exist_ok=True)

# 3. URL del mapa
url = "https://mapajaliscodesapariciones.rys2000dev.workers.dev/"

# 4. Obtener el HTML
print("Descargando HTML del mapa...")
response = requests.get(url, timeout=30)
html_content = response.text

# 5. Guardar HTML para análisis
with open(f'{drive_folder}/mapa_html.html', 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f"HTML guardado: {drive_folder}/mapa_html.html")

# 6. Buscar datos GeoJSON/JSON en el HTML
print("\n🔍 Buscando datos GeoJSON/JSON en el código...")

# Patrones para encontrar URLs de datos
patterns = [
    r'"url"\s*:\s*"([^"]+\.(?:geojson|json))"',
    r'src="([^"]+\.(?:geojson|json))"',
    r'data-url="([^"]+\.(?:geojson|json))"',
    r'load\(\s*[\'"]([^\'"]+\.(?:geojson|json))[\'"]',
    r'fetch\(\s*[\'"]([^\'"]+\.(?:geojson|json))[\'"]',
    r'L\.geoJSON\(\s*[\'"]([^\'"]+\.(?:geojson|json))[\'"]',
    r'addLayer\(\s*[\'"]([^\'"]+\.(?:geojson|json))[\'"]'
]

all_urls = []
for pattern in patterns:
    urls = re.findall(pattern, html_content, re.IGNORECASE)
    all_urls.extend(urls)

# Eliminar duplicados
unique_urls = list(set(all_urls))
print(f"Encontradas {len(unique_urls)} URLs potenciales de datos")

# 7. Analizar archivos JavaScript
print("\n🔍 Analizando archivos JavaScript...")
js_pattern = r'<script[^>]+src="([^"]+\.js)"'
js_urls = re.findall(js_pattern, html_content)

for js_url in js_urls[:5]:  # Analizar solo primeros 5 JS
    try:
        if not js_url.startswith('http'):
            js_url = url + js_url if js_url.startswith('/') else url + '/' + js_url

        print(f"  Analizando: {js_url}")
        js_response = requests.get(js_url, timeout=10)
        if js_response.status_code == 200:
            js_content = js_response.text

            # Buscar URLs de datos en JS
            for pattern in patterns:
                js_urls_found = re.findall(pattern, js_content, re.IGNORECASE)
                unique_urls.extend(js_urls_found)

    except Exception as e:
        print(f"  Error analizando {js_url}: {e}")

# 8. Descargar y procesar datos encontrados
print(f"\n📥 Descargando {len(set(unique_urls))} archivos de datos...")

all_data = []
downloaded_files = []

for i, data_url in enumerate(set(unique_urls)):
    try:
        # Completar URL si es relativa
        if not data_url.startswith('http'):
            if data_url.startswith('/'):
                data_url = 'https://mapajaliscodesapariciones.rys2000dev.workers.dev' + data_url
            else:
                data_url = url + '/' + data_url

        print(f"  {i+1}. Descargando: {data_url}")

        response = requests.get(data_url, timeout=15)

        if response.status_code == 200:
            # Intentar parsear como JSON/GeoJSON
            try:
                data = response.json()

                # Guardar archivo
                filename = f"data_{i+1}_{data_url.split('/')[-1]}"
                filepath = f'{drive_folder}/{filename}'

                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                downloaded_files.append({
                    'url': data_url,
                    'filename': filename,
                    'size_kb': len(response.content) / 1024
                })

                # Procesar datos según tipo
                if isinstance(data, dict):
                    if data.get('type') == 'FeatureCollection':
                        features = data.get('features', [])
                        print(f"    ✓ GeoJSON con {len(features)} features")

                        for feature in features:
                            if feature.get('geometry') and feature.get('properties'):
                                all_data.append(feature)

                    elif 'features' in data:
                        # Otro formato posible
                        features = data.get('features', [])
                        print(f"    ✓ Formato con {len(features)} features")

                        for feature in features:
                            all_data.append(feature)

                elif isinstance(data, list):
                    print(f"    ✓ Lista con {len(data)} elementos")
                    all_data.extend(data)

            except json.JSONDecodeError:
                print(f"    ✗ No es JSON válido")
                # Guardar igual por si acaso
                filename = f"data_{i+1}_{data_url.split('/')[-1]}"
                filepath = f'{drive_folder}/{filename}'
                with open(filepath, 'wb') as f:
                    f.write(response.content)

        else:
            print(f"    ✗ Error HTTP {response.status_code}")

    except Exception as e:
        print(f"    ✗ Error: {str(e)[:100]}")

# 9. Procesar y guardar datos extraídos
print(f"\n💾 Procesando {len(all_data)} registros extraídos...")

if all_data:
    # Convertir a DataFrame/GeoDataFrame
    processed_data = []

    for i, item in enumerate(all_data):
        try:
            record = {'id': i}

            if isinstance(item, dict):
                # Extraer propiedades
                if 'properties' in item:
                    record.update(item['properties'])

                # Extraer geometría si existe
                if 'geometry' in item and item['geometry']:
                    geom = item['geometry']
                    if geom.get('type') == 'Point' and geom.get('coordinates'):
                        coords = geom['coordinates']
                        record['longitud'] = coords[0]
                        record['latitud'] = coords[1]

                        # También guardar como texto
                        record['coordenadas'] = f"{coords[1]}, {coords[0]}"

                    elif geom.get('type') == 'Polygon' and geom.get('coordinates'):
                        # Para polígonos, usar el centroide
                        import numpy as np
                        coords_array = np.array(geom['coordinates'][0])
                        center = coords_array.mean(axis=0)
                        record['longitud_centro'] = float(center[0])
                        record['latitud_centro'] = float(center[1])

            processed_data.append(record)

        except Exception as e:
            print(f"  Error procesando registro {i}: {e}")

    # Crear DataFrame
    df = pd.DataFrame(processed_data)

    # Guardar en múltiples formatos
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    # CSV
    csv_path = f'{drive_folder}/datos_extraidos_{timestamp}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✅ CSV guardado: {csv_path}")

    # JSON
    json_path = f'{drive_folder}/datos_extraidos_{timestamp}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON guardado: {json_path}")

    # GeoJSON si hay coordenadas
    if 'latitud' in df.columns and 'longitud' in df.columns:
        # Filtrar filas con coordenadas válidas
        valid_coords = df.dropna(subset=['latitud', 'longitud'])

        if not valid_coords.empty:
            # Crear geometrías
            geometry = [Point(xy) for xy in zip(valid_coords['longitud'], valid_coords['latitud'])]

            # Crear GeoDataFrame
            gdf = gpd.GeoDataFrame(valid_coords, geometry=geometry, crs='EPSG:4326')

            # Guardar como GeoJSON
            geojson_path = f'{drive_folder}/datos_geo_{timestamp}.geojson'
            gdf.to_file(geojson_path, driver='GeoJSON')
            print(f"✅ GeoJSON guardado: {geojson_path}")

    # Resumen
    print(f"\n📊 RESUMEN:")
    print(f"  Total registros: {len(df)}")
    print(f"  Con coordenadas: {len(df.dropna(subset=['latitud', 'longitud'])) if 'latitud' in df.columns else 0}")
    print(f"  Columnas: {list(df.columns)[:10]}...")  # Mostrar primeras 10 columnas

else:
    print("⚠️ No se encontraron datos procesables")

# 10. Guardar reporte de archivos descargados
if downloaded_files:
    report_df = pd.DataFrame(downloaded_files)
    report_path = f'{drive_folder}/reporte_descargas_{timestamp}.csv'
    report_df.to_csv(report_path, index=False)
    print(f"\n📋 Reporte de descargas: {report_path}")

# 11. Método ALTERNATIVO: Buscar datos específicos en el HTML
print("\n🔍 Búsqueda alternativa: Analizando contenido HTML...")

# Buscar patrones específicos de datos en el HTML
data_patterns = {
    'coordenadas': r'[-+]?\d*\.\d+\s*,\s*[-+]?\d*\.\d+',
    'fechas': r'\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2}',
    'municipios': r'(?i)(Guadalajara|Zapopan|Tlaquepaque|Tonalá|Tlajomulco|El Salto|Ixtlahuacán)',
    'edades': r'(\d+)\s*años|\bEdad:\s*(\d+)',
}

html_data = {}
for key, pattern in data_patterns.items():
    matches = re.findall(pattern, html_content)
    if matches:
        html_data[key] = list(set(matches))[:50]  # Limitar a 50
        print(f"  {key}: {len(html_data[key])} encontrados")

# Guardar datos HTML
if html_data:
    html_data_path = f'{drive_folder}/datos_html_{timestamp}.json'
    with open(html_data_path, 'w', encoding='utf-8') as f:
        json.dump(html_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Datos HTML guardados: {html_data_path}")

print("\n" + "="*60)
print("¡ANÁLISIS COMPLETADO!")
print(f"📁 Archivos guardados en: {drive_folder}")
print("="*60)

# Listar archivos creados
print("\n📄 ARCHIVOS CREADOS:")
import glob
files = glob.glob(f'{drive_folder}/*')
for file in sorted(files)[-10:]:  # Mostrar últimos 10
    size = os.path.getsize(file) / 1024
    print(f"  • {os.path.basename(file)} ({size:.1f} KB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descargando HTML del mapa...
HTML guardado: /content/drive/My Drive/Desaparecidos_Jalisco/mapa_html.html

🔍 Buscando datos GeoJSON/JSON en el código...
Encontradas 3 URLs potenciales de datos

🔍 Analizando archivos JavaScript...
  Analizando: https://cdn.jsdelivr.net/npm/chart.js
  Analizando: https://unpkg.com/maplibre-gl@3.6.1/dist/maplibre-gl.js

📥 Descargando 3 archivos de datos...
  1. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//fosas.geojson
    ✓ GeoJSON con 128 features
  2. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//colonias.geojson
    ✓ GeoJSON con 6307 features
  3. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//municipios.geojson
    ✓ GeoJSON con 6624 features

💾 Procesando 13059 registros extraídos...
✅ CSV guardado: /content/drive/My Drive/Desaparecidos_Jalisco/dato